In [1]:
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 31.6 MB/s eta 0:00:00


In [2]:
import pandas as pd
import random
from faker import Faker
from datetime import datetime, timedelta
import numpy as np

In [3]:
fake = Faker()
data=[]

In [4]:
stages=['Browse','Add to Cart','Checkout','Purchase']
probabilities={
    'Browse':1.0,
    'Add to Cart':0.65,
    'Checkout':0.55,
    'Purchase':0.45
}

In [5]:
devices = np.random.choice(
    ['mobile', 'desktop', 'tablet'],
    size=50000,
    p=[0.72, 0.23, 0.05]
)


In [6]:
categories = np.random.choice(
    ['Electronics', 'Fashion', 'Home', 'Beauty', 'Sports'],
    size=50000,
    p=[0.30, 0.28, 0.18, 0.14, 0.10]
)

In [7]:
regions = np.random.choice(
    ['North', 'South', 'East', 'West'],
    size=50000,
    p=[0.35, 0.25, 0.20, 0.20]
)

In [8]:
channels = np.random.choice(
    ['Organic Search', 'Google Ads', 'Social Media', 'Email'],
    size=50000,
    p=[0.45, 0.25, 0.22, 0.08]
)

In [9]:
for i in range(50000):

    user_id = f"USR{i+1:05d}"
    session_id = f"SES{i+1:05d}"

    device = devices[i]
    channel = channels[i]
    category = categories[i]
    region = regions[i]

    event_time = fake.date_time_between(
        start_date='-30d',
        end_date='now'
    )

    # Base probabilities
    add_to_cart_prob = 0.22
    checkout_prob = 0.65
    purchase_prob = 0.72

    # Better-performing segments
    if device == "desktop":
        add_to_cart_prob += 0.06

    if channel == "Email":
        add_to_cart_prob += 0.08
        checkout_prob += 0.05

    if category == "Electronics":
        add_to_cart_prob += 0.05

    user_events = []

    # ----------------
    # Browse
    # ----------------
    user_events.append({
        'User_ID': user_id,
        'Session_ID': session_id,
        'Event': 'Browse',
        'Time_Stamp': event_time.strftime('%Y-%m-%d %H:%M:%S'),
        'Device': device,
        'Region': region,
        'Channel': channel,
        'Category': category,
        'Revenue': None
    })

    # ----------------
    # Add To Cart
    # ----------------
    if random.random() < add_to_cart_prob:

        event_time += timedelta(
            minutes=random.randint(1, 8)
        )

        user_events.append({
            'User_ID': user_id,
            'Session_ID': session_id,
            'Event': 'Add to Cart',
            'Time_Stamp': event_time.strftime('%Y-%m-%d %H:%M:%S'),
            'Device': device,
            'Region': region,
            'Channel': channel,
            'Category': category,
            'Revenue': None
        })

        # ----------------
        # Checkout
        # ----------------
        if random.random() < checkout_prob:

            event_time += timedelta(
                minutes=random.randint(2, 15)
            )

            user_events.append({
                'User_ID': user_id,
                'Session_ID': session_id,
                'Event': 'Checkout',
                'Time_Stamp': event_time.strftime('%Y-%m-%d %H:%M:%S'),
                'Device': device,
                'Region': region,
                'Channel': channel,
                'Category': category,
                'Revenue': None
            })

            # ----------------
            # Purchase
            # ----------------
            if random.random() < purchase_prob:

                event_time += timedelta(
                    minutes=random.randint(1, 10)
                )

                revenue = round(
                    np.random.lognormal(
                        mean=6.0,
                        sigma=0.7
                    ),
                    2
                )

                revenue = min(revenue, 15000)

                user_events.append({
                    'User_ID': user_id,
                    'Session_ID': session_id,
                    'Event': 'Purchase',
                    'Time_Stamp': event_time.strftime('%Y-%m-%d %H:%M:%S'),
                    'Device': device,
                    'Region': region,
                    'Channel': channel,
                    'Category': category,
                    'Revenue': revenue
                })

    # Bounce = only Browse event
    bounce = len(user_events) == 1

    for record in user_events:
        record['Bounce_back'] = bounce

    data.extend(user_events)

In [10]:
df = pd.DataFrame(data)

print(df.shape)
df.head()

(77051, 10)


,User_ID,Session_ID,Event,Time_Stamp,Device,Region,Channel,Category,Revenue,Bounce_back
0,USR00001,SES00001,Browse,2026-06-01 02:32:19,mobile,North,Organic Search,Fashion,NaN,False
1,USR00001,SES00001,Add to Cart,2026-06-01 02:33:19,mobile,North,Organic Search,Fashion,NaN,False
2,USR00001,SES00001,Checkout,2026-06-01 02:41:19,mobile,North,Organic Search,Fashion,NaN,False
3,USR00001,SES00001,Purchase,2026-06-01 02:48:19,mobile,North,Organic Search,Fashion,128.53,False
4,USR00002,SES00002,Browse,2026-06-01 08:34:26,desktop,North,Social Media,Home,NaN,True


In [11]:
df.to_csv("user_journey_funnel.csv", index=False)

In [12]:
df[df['Event']=='Purchase']

,User_ID,Session_ID,Event,Time_Stamp,Device,Region,Channel,Category,Revenue,Bounce_back
3,USR00001,SES00001,Purchase,2026-06-01 02:48:19,mobile,North,Organic Search,Fashion,128.53,False
10,USR00005,SES00005,Purchase,2026-06-04 00:04:16,desktop,East,Organic Search,Fashion,289.24,False
15,USR00007,SES00007,Purchase,2026-05-18 00:21:07,desktop,West,Email,Electronics,412.24,False
41,USR00026,SES00026,Purchase,2026-06-12 05:30:14,mobile,West,Social Media,Home,691.94,False
54,USR00035,SES00035,Purchase,2026-05-31 00:32:38,mobile,North,Social Media,Electronics,744.81,False
...,...,...,...,...,...,...,...,...,...,...
76965,USR49938,SES49938,Purchase,2026-05-17 14:18:08,mobile,South,Google Ads,Fashion,484.07,False
76987,USR49953,SES49953,Purchase,2026-06-13 08:33:45,mobile,East,Organic Search,Beauty,313.27,False
77006,USR49967,SES49967,Purchase,2026-06-12 05:35:01,desktop,North,Google Ads,Beauty,198.84,False
77015,USR49972,SES49972,Purchase,2026-06-09 05:38:27,mobile,North,Organic Search,Home,564.65,False
